In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import requests
import tqdm

In [ ]:
genome_index = pd.read_csv("/data/MIRIPVIR25/sanchis21/genomes-index.csv", header=None, names=['id', 'species_name'], sep=';')
genome_index['species_name'] = genome_index['species_name'].apply(lambda x: x.replace(',', ''))
genome_index = genome_index.drop_duplicates(subset=['id', 'species_name'], keep='first')
genome_index['is_MAG'] = genome_index['species_name'].apply(lambda x: x.find('MAG') != -1)
genome_index = genome_index.query('is_MAG == False')
genome_index

In [ ]:
genome_index.species_name.unique()

In [ ]:
def get_taxid(name):
    print(name)
    r = requests.get(f"https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/taxon/{name}")
    try:
        r.raise_for_status()
        data = r.json()
    except requests.HTTPError:
        print(f"-- taxon {name}, not found")
        return None
    
    try:
        return data['taxonomy_nodes'][0]['taxonomy']['tax_id']
    except KeyError:
        return "not found"

In [ ]:
taxonomy = pd.DataFrame.from_records([dict(name=name, taxid=get_taxid(name)) for name in genome_index.species_name.unique()])
taxonomy[:10]

In [ ]:
genome_index = pd.merge(genome_index, taxonomy, left_on='species_name', right_on='name')[['id', 'species_name', 'taxid']]
genome_index[:10]

In [ ]:
genome_index.to_json('../data/sanchis21.genome-index.json', orient='records', indent=4)